# BasePromptTemplate: `RunnableSerializable[dict[str, Any], PromptValue]`, `ABC`, `Generic[FormatOutputType]`

`BasePromptTemplate` is an abstract, serializable Runnable that defines the common interface and behaviour for prompt templates.

It accepts a dictionary of input variables, validates them, formats the prompt, and returns a `PromptValue`.

In [1]:
from langchain_core.prompts import BasePromptTemplate, PromptTemplate


prompt: BasePromptTemplate = PromptTemplate.from_template(
    "Explain {topic} in {style}." # Prompt containing required variables
)

result = prompt.invoke(
    {
        "topic": "inheritance", # Value for the topic variable
        "style": "one sentence" # Value for the style variable
    }
)

print("Prompt class:", type(prompt).__name__) # Concrete prompt class
print("Is BasePromptTemplate:", isinstance(prompt, BasePromptTemplate)) # True
print("Input variables:", prompt.input_variables) # Required variables
print("Result type:", type(result).__name__) # PromptValue type
print("Formatted prompt:", result.to_string()) # Final formatted text

Prompt class: PromptTemplate
Is BasePromptTemplate: True
Input variables: ['style', 'topic']
Result type: StringPromptValue
Formatted prompt: Explain inheritance in one sentence.


# Fields

1. `input_variables`:`list[str]`:= Stores the names of variables that must be supplied when formatting the prompt.

2. `optional_variables`:`list[str]`:= Stores automatically inferred placeholder variables that users are not required to provide. Its default value is an empty list.

3. `input_types`:`dict[str, Any]`:= Maps input-variable names to their expected types. Variables without an explicitly declared type are treated as strings. Its default value is an empty dictionary.

4. `output_parser`:`BaseOutputParser | None`:= Stores an optional parser used to process the output generated after the prompt is passed to a language model. Its default value is `None`.

5. `partial_variables`:`Mapping[str, Any]`:= Stores variables already assigned to the prompt so they do not need to be supplied during every call. Its default value is an empty mapping.

6. `metadata`:`dict[str, Any] | None`:= Stores metadata added to tracing information when the prompt is invoked. Its default value is `None`.

7. `tags`:`list[str] | None`:= Stores tags added to tracing information when the prompt is invoked. Its default value is `None`.

## Configuration

1. `model_config`:`ConfigDict`:= Allows arbitrary Python types to be used in the Pydantic model.

   ```python
   model_config = ConfigDict(
       arbitrary_types_allowed=True # Allow arbitrary Python types
   )
   ```

## Properties

1. `OutputType`:`StringPromptValue | ChatPromptValueConcrete`:= Returns the possible prompt-value types produced when the prompt is invoked.

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate


prompt = PromptTemplate(
    template="Explain {topic} in a {style} way.", # Prompt template
    input_variables=["topic"], # Required variable
    optional_variables=[], # Variables that are not required
    input_types={
        "topic": str
    }, # Expected input-variable types
    output_parser=StrOutputParser(), # Optional output parser
    partial_variables={
        "style": "simple"
    }, # Preassigned variable
    metadata={
        "chapter": "prompt-base"
    }, # Tracing metadata
    tags=[
        "prompt",
        "example"
    ] # Tracing tags
)


result = prompt.invoke(
    {
        "topic": "inheritance"
    } # Only the required variable is supplied
)


print("Input variables:", prompt.input_variables)
print("Optional variables:", prompt.optional_variables)
print("Input types:", prompt.input_types)
print("Output parser:", type(prompt.output_parser).__name__)
print("Partial variables:", prompt.partial_variables)
print("Metadata:", prompt.metadata)
print("Tags:", prompt.tags)

print(
    "Arbitrary types allowed:",
    prompt.model_config["arbitrary_types_allowed"]
) # Display model configuration

print("OutputType:", prompt.OutputType) # Possible PromptValue output type
print("Formatted prompt:", result.to_string()) # Display formatted prompt

Input variables: ['topic']
Optional variables: []
Input types: {'topic': <class 'str'>}
Output parser: StrOutputParser
Partial variables: {'style': 'simple'}
Metadata: {'chapter': 'prompt-base'}
Tags: ['prompt', 'example']
Arbitrary types allowed: True
OutputType: langchain_core.prompt_values.StringPromptValue | langchain_core.prompt_values.ChatPromptValueConcrete
Formatted prompt: Explain inheritance in a simple way.


# Methods

1. `validate_variable_names`:= Validates the input and partial variable names.

   It ensures that `stop` is not used as an input or partial variable and that input variables do not overlap with partial variables.

   ```python
   validate_variable_names(
       self
   ) -> Self
   ```

2. `get_lc_namespace`:= Returns the LangChain serialization namespace used for prompt templates.

   ```python
   @classmethod
   get_lc_namespace(
       cls # Prompt-template class
   ) -> list[str]
   ```

3. `is_lc_serializable`:= Indicates whether prompt templates support LangChain serialization.

   ```python
   @classmethod
   is_lc_serializable(
       cls # Prompt-template class
   ) -> bool
   ```

4. `get_input_schema`:= Creates a Pydantic model containing the required and optional prompt input variables.

   ```python
   get_input_schema(
       self,
       config: RunnableConfig | None = None # Configuration used to generate the schema
   ) -> type[BaseModel]
   ```

5. `invoke`:= Synchronously validates the input, adds prompt metadata and tags to the runtime configuration, and returns the formatted `PromptValue`.

   ```python
   invoke(
       self,
       input: dict[str, Any], # Input variables passed to the prompt
       config: RunnableConfig | None = None, # Runtime prompt configuration
       **kwargs: Any # Additional invocation arguments
   ) -> PromptValue
   ```

6. `ainvoke`:= Asynchronously validates the input, adds prompt metadata and tags to the runtime configuration, and returns the formatted `PromptValue`.

   ```python
   async ainvoke(
       self,
       input: dict[str, Any], # Input variables passed to the prompt
       config: RunnableConfig | None = None, # Runtime prompt configuration
       **kwargs: Any # Additional invocation arguments
   ) -> PromptValue
   ```

7. `format_prompt`:= Abstract method that creates a `PromptValue` using the supplied variables.

   ```python
   format_prompt(
       self,
       **kwargs: Any # Variables used to format the prompt
   ) -> PromptValue
   ```

8. `aformat_prompt`:= Asynchronously creates a `PromptValue` using the supplied variables.

   ```python
   async aformat_prompt(
       self,
       **kwargs: Any # Variables used to format the prompt
   ) -> PromptValue
   ```

9. `partial`:= Creates a new prompt template with selected variables already assigned.

   Partial values may be fixed strings or callable functions that return strings.

   ```python
   partial(
       self,
       **kwargs: str | Callable[[], str] # Fixed values or callable partial variables
   ) -> BasePromptTemplate[FormatOutputType]
   ```

10. `format`:= Abstract method that formats the prompt using the supplied variables and returns the template-specific output type.

```python
format(
    self,
    **kwargs: Any # Variables used to format the prompt
) -> FormatOutputType
```

11. `aformat`:= Asynchronously formats the prompt using the supplied variables.

```python
async aformat(
    self,
    **kwargs: Any # Variables used to format the prompt
) -> FormatOutputType
```

12. `dict`:= Returns a dictionary representation of the prompt.

This method is deprecated. Use `asdict` instead.

```python
dict(
    self,
    **kwargs: Any # Additional dictionary-generation options
) -> dict[str, Any]
```

13. `asdict`:= Returns a dictionary representation of the prompt and includes its prompt-type key when available.

```python
asdict(
    self,
    **kwargs: Any # Additional dictionary-generation options
) -> dict[str, Any]
```

14. `save`:= Saves the prompt as a JSON, YAML, or YML file.

This method is deprecated in favour of LangChain load and dump utilities.

```python
save(
    self,
    file_path: Path | str # Destination JSON, YAML, or YML file
) -> None
```

In [4]:
import asyncio
import warnings

from langchain_core.prompts import PromptTemplate


async def main() -> None:
    prompt = PromptTemplate(
        template="Explain {topic} in a {style} way.", # Prompt template
        input_variables=["topic", "style"], # Required variables
        metadata={"chapter": "prompts"}, # Prompt tracing metadata
        tags=["base-prompt"] # Prompt tracing tags
    )


    # 1. Validate variable names
    validated_prompt = prompt.validate_variable_names()

    print("Validated:", validated_prompt is prompt)


    # 2. Get the LangChain serialization namespace
    namespace = PromptTemplate.get_lc_namespace()

    print("Namespace:", namespace)


    # 3. Check serialization support
    serializable = PromptTemplate.is_lc_serializable()

    print("Serializable:", serializable)


    # 4. Create the Pydantic input schema
    input_schema = prompt.get_input_schema(
        config=None # Optional Runnable configuration
    )

    print("Input schema:")
    print(input_schema.model_json_schema())


    # 5. Invoke the prompt synchronously
    invoked_value = prompt.invoke(
        {
            "topic": "inheritance",
            "style": "simple"
        }, # Prompt input variables
        config={
            "tags": ["synchronous"],
            "metadata": {"source": "example"}
        } # Runtime configuration
    )

    print("invoke():", invoked_value.to_string())


    # 6. Invoke the prompt asynchronously
    async_invoked_value = await prompt.ainvoke(
        {
            "topic": "polymorphism",
            "style": "concise"
        }, # Prompt input variables
        config={
            "tags": ["asynchronous"]
        } # Runtime configuration
    )

    print("ainvoke():", async_invoked_value.to_string())


    # 7. Create a PromptValue synchronously
    prompt_value = prompt.format_prompt(
        topic="encapsulation", # Value for topic
        style="beginner-friendly" # Value for style
    )

    print("format_prompt():", prompt_value.to_string())


    # 8. Create a PromptValue asynchronously
    async_prompt_value = await prompt.aformat_prompt(
        topic="abstraction", # Value for topic
        style="detailed" # Value for style
    )

    print("aformat_prompt():", async_prompt_value.to_string())


    # 9. Preassign a variable
    partial_prompt = prompt.partial(
        style=lambda: "simple" # Callable partial variable
    )

    print(
        "partial():",
        partial_prompt.format(
            topic="method overriding" # Only unassigned variable required
        )
    )


    # 10. Format the prompt synchronously
    formatted_text = prompt.format(
        topic="classes", # Value for topic
        style="short" # Value for style
    )

    print("format():", formatted_text)


    # 11. Format the prompt asynchronously
    async_formatted_text = await prompt.aformat(
        topic="objects", # Value for topic
        style="clear" # Value for style
    )

    print("aformat():", async_formatted_text)


    # 12. Get a dictionary using deprecated dict()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        deprecated_dictionary = prompt.dict(
            exclude_none=True # Exclude fields containing None
        )

    print("dict():", deprecated_dictionary)


    # 13. Get the recommended dictionary representation
    prompt_dictionary = prompt.asdict(
        exclude_none=True # Exclude fields containing None
    )

    print("asdict():", prompt_dictionary)


    # 14. Save the non-partial prompt
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prompt.save(
            "saved_prompt.json" # Destination JSON file
        )

    print("save(): saved_prompt.json")


await main() # Correct for Jupyter Notebook

Validated: True
Namespace: ['langchain', 'prompts', 'prompt']
Serializable: True
Input schema:
{'properties': {'style': {'title': 'Style', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}}, 'required': ['style', 'topic'], 'title': 'PromptInput', 'type': 'object'}
invoke(): Explain inheritance in a simple way.
ainvoke(): Explain polymorphism in a concise way.
format_prompt(): Explain encapsulation in a beginner-friendly way.
aformat_prompt(): Explain abstraction in a detailed way.
partial(): Explain method overriding in a simple way.
format(): Explain classes in a short way.
aformat(): Explain objects in a clear way.
dict(): {'input_variables': ['style', 'topic'], 'optional_variables': [], 'partial_variables': {}, 'metadata': {'chapter': 'prompts'}, 'tags': ['base-prompt'], 'template': 'Explain {topic} in a {style} way.', 'template_format': 'f-string', 'validate_template': False, '_type': 'prompt'}
asdict(): {'input_variables': ['style', 'topic'], 'optional_variables': [

# Functions

1. `format_document`:= Synchronously formats a `Document`.

   It passes the document's `page_content` and required metadata fields into the supplied prompt template.

   ```python
   format_document(
       doc: Document, # Document containing page content and metadata
       prompt: BasePromptTemplate[str] # Prompt used to format the document
   ) -> str
   ```

2. `aformat_document`:= Asynchronously formats a `Document`.

   It passes the document's `page_content` and required metadata fields into the supplied prompt template.

   ```python
   async aformat_document(
       doc: Document, # Document containing page content and metadata
       prompt: BasePromptTemplate[str] # Prompt used to format the document
   ) -> str
   ```

In [5]:
from langchain_core.documents import Document
from langchain_core.prompts import (
    PromptTemplate,
    aformat_document,
    format_document,
)


document = Document(
    page_content="LangChain simplifies LLM application development.", # Document content
    metadata={"source": "LangChain notes"} # Document metadata
)

prompt = PromptTemplate.from_template(
    "Source: {source}\nContent: {page_content}" # Uses content and metadata
)


sync_result = format_document(
    document, # Document to format
    prompt # Formatting prompt
)

async_result = await aformat_document(
    document, # Document to format
    prompt # Formatting prompt
)


print("Synchronous:\n", sync_result)
print("\nAsynchronous:\n", async_result)

Synchronous:
 Source: LangChain notes
Content: LangChain simplifies LLM application development.

Asynchronous:
 Source: LangChain notes
Content: LangChain simplifies LLM application development.
